# P1: Seismic Cross-Correlation (Provenance)

**Provenance track.** These notebooks document how the shipped data files
were made; they are not needed to reproduce the paper's results. This one
requires live access to the IRIS data service for station ER.CON waveforms.

Builds a *master* seismic waveform by stacking the verified large eruptions
listed in `data/MasterList.txt` (117 events, December 2013 to December 2014),
then cross-correlates it against continuous seismic data to find candidate
large eruptions. The catalog and the template were grown iteratively:
correlate, verify new hits against the imagery, add them to the list,
restack (Dye & Morra 2020); the shipped file is the final state of that
iteration.

> **Manual step (final output already provided).** The raw correlation
> hits below fire on *every sample* above threshold, so one eruption
> produces many consecutive hits, and weak non-eruptive long-period signals
> can also cross 0.10. The published catalog was produced by manually
> deduplicating and verifying each hit against the infrared imagery (an
> amplitude floor of 10,000 counts removed four spurious low-amplitude
> events; see Dye & Morra 2020 §3.2). The verified seed list ships as
> `data/MasterList.txt`; the verified eruption *frames* are already in the
> training dataset. You do not need to re-run this.

In [ ]:
# --- Repository setup -------------------------------------------------------
# Locates config.py at the repository root so this notebook runs no matter
# which directory Jupyter was launched from. All paths and constants live in
# config.py; edit that one file rather than anything below.
import sys
from pathlib import Path

_root = Path.cwd().resolve()
while not (_root / "config.py").exists():
    if _root.parent == _root:
        raise FileNotFoundError(
            "config.py not found. Launch Jupyter from inside the repository, "
            "or place this notebook back in its original folder.")
    _root = _root.parent
sys.path.insert(0, str(_root))

import config
config.describe()


In [ ]:
# Requires: pip install obspy
from obspy.clients.fdsn import Client
from obspy import UTCDateTime
from obspy.signal.cross_correlation import correlate
import matplotlib.pyplot as plt
import numpy as np

client = Client("IRIS")

NETWORK, STATION, CHANNEL = "ER", "CON", "BH2"
MASTER_HALF_WINDOW_S = 12.5      # master = +/- 12.5 s around each seed time
CORRELATION_THRESHOLD = 0.10
EDGE_CLIP_SAMPLES = 2401         # 60 s at 40 Hz: ends too short to correlate
SAMPLE_RATE = 40                 # Hz (ER.CON broadband)

with open(config.MASTERLIST_PATH) as f:
    master_events = [line.strip() for line in f if line.strip()]
print(f"{len(master_events)} catalog eruptions loaded from MasterList.txt (expected 117)")


## 1. Stack the master waveform

Each seed's ±12.5 s trace is mean-centered and summed; the average is the
template whose shape a candidate eruption must match.

In [ ]:
stack = 0
for event in master_events:
    t = UTCDateTime(event)
    trace = client.get_waveforms(NETWORK, STATION, "--", CHANNEL,
                                 t - MASTER_HALF_WINDOW_S, t + MASTER_HALF_WINDOW_S)
    centered = trace[0].data - trace[0].data.mean()
    stack = stack + centered
    plt.plot(centered, lw=0.6)
plt.title("Individual seed waveforms (mean-centered)"); plt.show()

master = stack / len(master_events)
plt.plot(master); plt.title("Stacked master waveform"); plt.show()


## 2. Cross-correlate a continuous window

Pick any period; every sample whose normalized correlation with the master
exceeds 0.10 is written to `outputs/Correlated_Events.txt`. Expect long
requests to be slow: a two-week waveform pull from IRIS is hundreds of MB.

In [ ]:
START = UTCDateTime("2014-01-01T00:00:00")   # edit as needed
END   = UTCDateTime("2014-01-15T00:00:00")

MAX_PLOTS   = 10     # cap on correlation figures (one per continuous segment)
GROUP_GAP_S = 60     # consecutive hits closer than this belong to one candidate

station = client.get_waveforms(NETWORK, STATION, "--", CHANNEL, START, END)
print(station)
station.plot(size=(1100, 320))

out_path = config.OUTPUT_DIR / "Correlated_Events.txt"
count = plotted = 0
candidates = 0

with open(out_path, "w") as out:
    for k, trace in enumerate(station):
        if trace.data.size <= 12000:    # skip fragments under ~5 minutes
            print(f"segment {k}: {trace.data.size:,} samples (<5 min) - skipped")
            continue

        series = correlate(master[-1:0:-1], trace.data[-1:0:-1],
                           int(len(trace.data) / 2))
        series = np.asarray(series)

        pos = np.arange(1, len(series) + 1)
        lo, hi = EDGE_CLIP_SAMPLES, trace.data.size - EDGE_CLIP_SAMPLES
        valid = (pos >= lo) & (pos < hi)        # inclusive lower bound
        hits = valid & (series > CORRELATION_THRESHOLD)

        for p, v in zip(pos[hits], series[hits]):
            count += 1
            out.write(f"{count} Time: {trace.stats.starttime + p / SAMPLE_RATE}"
                      f" Correlation: {v}\n")

        # Collapse runs of consecutive hits into candidate eruptions
        hit_s = pos[hits] / SAMPLE_RATE
        n_cand = 0 if hit_s.size == 0 else 1 + int(
            (np.diff(hit_s) > GROUP_GAP_S).sum())
        candidates += n_cand
        print(f"segment {k}: {trace.stats.starttime} to {trace.stats.endtime}  "
              f"| {trace.data.size:,} samples | max r = {series.max():.3f} "
              f"| {int(hits.sum())} hits in {n_cand} candidate(s)")

        if plotted < MAX_PLOTS:
            t_s = pos / SAMPLE_RATE
            fig, ax = plt.subplots(figsize=(12, 3))
            ax.plot(t_s, series, lw=0.4, color="#333333")
            ax.axhline(CORRELATION_THRESHOLD, color="firebrick", ls="--", lw=1,
                       label=f"threshold {CORRELATION_THRESHOLD}")
            ax.axvspan(t_s[0], lo / SAMPLE_RATE, color="gray", alpha=0.30)
            ax.axvspan(hi / SAMPLE_RATE, t_s[-1], color="gray", alpha=0.30,
                       label=f"edge clip ({EDGE_CLIP_SAMPLES} samples)")
            if hits.any():
                ax.plot(hit_s, series[hits], "o", ms=3, color="firebrick",
                        label=f"{int(hits.sum())} samples above threshold")
            ax.set_xlim(t_s[0], t_s[-1])
            ax.set_xlabel(f"seconds after {trace.stats.starttime}")
            ax.set_ylabel("normalized correlation")
            ax.set_title(f"segment {k}: cross-correlation with the master waveform")
            ax.legend(loc="upper right", fontsize=8)
            plt.tight_layout()
            plt.show()
            plotted += 1

print(f"\n{count} raw sample-level hits in {candidates} candidate eruption(s) "
      f"-> {out_path}")
if plotted == MAX_PLOTS:
    print(f"(correlation plots capped at {MAX_PLOTS}; raise MAX_PLOTS for more)")
print("Candidates are NOT a catalog: each still needs visual verification "
      "against the infrared frames. data/MasterList.txt is the verified result "
      "of that process for December 2013 - December 2014.")